In [12]:
import os
from pprint import pprint

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [30]:
ROOT_DIR = "data"
RANDOM_STATE = 110

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
train_data

,Wip Line_Dam,Process Desc._Dam,Equipment_Dam,Model.Suffix_Dam,Workorder_Dam,Insp. Seq No._Dam,Insp Judge Code_Dam,CURE END POSITION X Collect Result_Dam,CURE END POSITION X Unit Time_Dam,CURE END POSITION X Judge Value_Dam,...,Production Qty Collect Result_Fill2,Production Qty Unit Time_Fill2,Production Qty Judge Value_Fill2,Receip No Collect Result_Fill2,Receip No Unit Time_Fill2,Receip No Judge Value_Fill2,WorkMode Collect Result_Fill2,WorkMode Unit Time_Fill2,WorkMode Judge Value_Fill2,target
0,IVI-OB6,Dam Dispenser,Dam dispenser #1,AJX75334505,4F1XA938-1,1,OK,240.0,NaN,NaN,...,7,NaN,NaN,127,NaN,NaN,1,NaN,NaN,Normal
1,IVI-OB6,Dam Dispenser,Dam dispenser #1,AJX75334505,3KPM0016-2,1,OK,240.0,NaN,NaN,...,185,NaN,NaN,1,NaN,NaN,0,NaN,NaN,Normal
2,IVI-OB6,Dam Dispenser,Dam dispenser #2,AJX75334501,4E1X9167-1,1,OK,1000.0,NaN,NaN,...,10,NaN,NaN,73,NaN,NaN,1,NaN,NaN,Normal
3,IVI-OB6,Dam Dispenser,Dam dispenser #2,AJX75334501,3K1X0057-1,1,OK,1000.0,NaN,NaN,...,268,NaN,NaN,1,NaN,NaN,0,NaN,NaN,Normal
4,IVI-OB6,Dam Dispenser,Dam dispenser #1,AJX75334501,3HPM0007-1,1,OK,240.0,NaN,NaN,...,121,NaN,NaN,1,NaN,NaN,0,NaN,NaN,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40501,IVI-OB6,Dam Dispenser,Dam dispenser #1,AJX75334501,3J1XF434-2,1,OK,240.0,NaN,NaN,...,318,NaN,NaN,1,NaN,NaN,0,NaN,NaN,Normal
40502,IVI-OB6,Dam Dispenser,Dam dispenser #2,AJX75334501,4E1XC796-1,1,OK,1000.0,NaN,NaN,...,14,NaN,NaN,197,NaN,NaN,1,NaN,NaN,Normal
40503,IVI-OB6,Dam Dispenser,Dam dispenser #1,AJX75334501,4C1XD438-1,1,OK,240.0,NaN,NaN,...,1,NaN,NaN,27,NaN,NaN,1,NaN,NaN,Normal
40504,IVI-OB6,Dam Dispenser,Dam dispenser #2,AJX75334501,3I1XA258-1,1,OK,1000.0,NaN,NaN,...,117,NaN,NaN,1,NaN,NaN,0,NaN,NaN,Normal


In [3]:
pip install catboost

Defaulting to user installation because normal site-packages is not writeable
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/165.9 kB ? eta -:--:--
     ------------------------------------  163.8/165.9 kB 10.2 MB/s eta 0:00:01
     -------------------------------------- 165.9/165.9 kB 2.5 MB/s eta 0:00:00
  Using cached pyparsing-3.1.2-py3-none-any.whl.metadata (5.1 kB)
   ---------------------------------------- 0.0/101.1 MB ? eta -:--:--
   ---------------------------------------- 0.6/101.1 MB 18.5 MB/s eta 0:00:06
   ---------------------------------------- 1.2/101.1 MB 15.8 MB/s eta 0:00:07
    --------------------------------------- 2.1/101.1 MB 16.6 MB/s eta 0:00:06
   - -------------------------------------- 2.5/101.1 MB 14.6 MB/s eta 0:00:07
   - -------------------------------------- 3.0/101.1 MB 13.9 MB/s eta 0:00:08
   - -------------------------------------- 3.9/101.1 MB 14.6 MB/s eta 0:00:07
   - ---------


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/258.3 kB ? eta -:--:--
   ---------------------------------------- 258.3/258.3 kB 5.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

# 데이터 불러오기 (예: CSV 파일로부터 불러오기)
# data = pd.read_csv('your_data.csv')

# X, y 분리
X = train_data.drop(columns=['target'])  # 특성(피처)
y = train_data['target']  # 타겟 ('Normal', 'Abnormal')

# 카테고리형 컬럼 식별 (예: 문자열 또는 범주형 데이터)
cat_features = X.select_dtypes(include=['object']).columns.tolist()


CatBoost로 카테고리형 데이터 처리

In [32]:
# NaN 값이 있는 행을 제거
# X = X.dropna(subset=cat_features)

# NaN 값 처리 후 X와 y의 인덱스 동기화
X = X[cat_features].fillna('missing')  # 카테고리형 데이터 NaN 값 처리
y = y.loc[X.index]  # X의 인덱스와 y의 인덱스를 맞추기

# 다시 한번 X와 y의 크기를 확인
assert len(X) == len(y), "X와 y의 크기가 일치하지 않습니다."

# CatBoost 모델 초기화
catboost_model = CatBoostClassifier(iterations=100, depth=3, learning_rate=0.1, loss_function='Logloss', cat_features=cat_features, verbose=0)

# CatBoost 모델을 사용해 피처 엔지니어링
catboost_model.fit(X, y)

# 학습된 임베딩 또는 피처를 얻기 위해 Pool 사용
train_pool = Pool(data=X, label=y, cat_features=cat_features)
X_catboost = catboost_model.predict_proba(train_pool)

# X_catboost는 이제 각 샘플의 새로운 피처 벡터 (예측 확률)입니다.
# X_transformed = pd.DataFrame(X_catboost, index=X.index)  


Isolation Forest로 이상치 탐지

In [33]:
# Isolation Forest로 이상치 탐지
iso_forest = IsolationForest(contamination='auto', random_state=42)
y_pred_iso = iso_forest.fit_predict(X_catboost)

# 이상치 필터링 (이상치 제거)
X_filtered = X_catboost[y_pred_iso == 1]
y_filtered = y[y_pred_iso == 1]

SVM 사용한 분류

In [37]:
# SVM 모델로 최종 분류 수행
svm_model = SVC(kernel='rbf', class_weight='balanced', probability=True)
svm_model.fit(X_filtered, y_filtered)

SVC(class_weight='balanced', probability=True)

학습 데이터에 적용된 것과 동일한 전처리 및 피처 엔지니어링을 테스트 데이터에도 적용

In [38]:
import pandas as pd
from catboost import Pool

# 테스트 데이터 로드
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

# 학습 데이터에서 사용된 피처를 그대로 사용 (train_data에서 사용된 features)
df_test_x = test_data[X.columns]  # train_data의 X와 동일한 피처 사용

# 테스트 데이터에 동일한 전처리 적용
for col in df_test_x.columns:
    if col in cat_features:
        df_test_x[col] = df_test_x[col].fillna('missing')  # NaN 값 처리

# CatBoost 피처 엔지니어링 (학습된 모델 사용)
test_pool = Pool(data=df_test_x, cat_features=cat_features)
df_test_x_catboost = catboost_model.predict_proba(test_pool)

# Isolation Forest로 이상치 탐지 (학습된 모델 사용)
test_pred_iso = iso_forest.predict(df_test_x_catboost)

# 이상치 필터링 (이상치 제거)
df_test_x_filtered = df_test_x_catboost[test_pred_iso == 1]

# SVM 모델로 최종 예측 수행 (학습된 모델 사용)
test_pred = svm_model.predict(df_test_x_filtered)

# 예측 결과 출력
print(test_pred)


C:\Users\user\AppData\Local\Temp\ipykernel_9260\353891927.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_x[col] = df_test_x[col].fillna('missing')  # NaN 값 처리


['Normal' 'Normal' 'Normal' ... 'Normal' 'AbNormal' 'Normal']


In [40]:
# SVM 모델로 이상치 필터링 없이 전체 데이터에 대해 예측 수행
test_pred = svm_model.predict(df_test_x_catboost)

# 제출 데이터에 예측값 반영
df_sub = pd.read_csv("submission_origin.csv")
df_sub["target"] = test_pred

# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)

별도로) test 데이터파일 따로 사용하지 않을 경우

In [34]:
# 데이터셋을 학습/테스트 세트로 분리
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.3, random_state=42, stratify=y_filtered)

# SMOTE를 사용하여 소수 클래스를 오버샘플링
#smote = SMOTE(random_state=42)
#X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


SVM 사용한 분류

In [35]:
# SVM 모델로 최종 분류 수행
svm_model = SVC(kernel='rbf', class_weight='balanced', probability=True)
svm_model.fit(X_train, y_train)

# 테스트 데이터 예측
y_pred = svm_model.predict(X_test)

# 결과 평가
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[ 151  246]
 [1727 7354]]
              precision    recall  f1-score   support

    AbNormal       0.08      0.38      0.13       397
      Normal       0.97      0.81      0.88      9081

    accuracy                           0.79      9478
   macro avg       0.52      0.60      0.51      9478
weighted avg       0.93      0.79      0.85      9478



In [36]:
y_pred

array(['Normal', 'Normal', 'AbNormal', ..., 'Normal', 'Normal', 'Normal'],
      dtype=object)